# Ders 0 — PyTorch Giriş: Kurulum, Tensor ve İlk Hesap
**Program: Python ile Derin Öğrenme Tabanlı Görüntü ve Doğal Dil İşleme** · **Modül 0 · Hazırlık Oturumu** · süre 2 saat

> Ders 1'den **önce** çalıştırılacak hazırlık notudur. Amaç: PyTorch'un kurulu ve çalışır olduğundan emin olmak ve tensor ile otomatik türev fikrine ilk el sıkışmayı yapmadan Ders 1'e gelmek.

## Bu oturumun hedefleri

1. PyTorch'un ne olduğunu ve derin öğrenme çerçevesi (framework) olarak neyi kolaylaştırdığını tek cümleyle açıklar.
2. Ortamını doğrular: PyTorch sürümü, GPU var/yok durumu ve CPU yedeği.
3. Skaler–vektör–matris–tensor hiyerarşisini `torch` ile kurar; `shape` ve `dtype`'ı okur.
4. Tensor aritmetiğini, broadcasting kuralını ve matris çarpımını elle hesapla doğrular.
5. Numpy ↔ Torch geçişini yapar ve `autograd` ile ilk otomatik türevi hesaplar.

> **Nasıl çalışılır:** Hücreleri sırayla `Shift+Enter` ile çalıştır. Her kod hücresi kendi başına çalışır ve ekrana anlaşılır bir çıktı basar. Sarı kutu notu: her satırın açıklaması hücrenin içindeki `#` yorumlarında — satır satır okumak serbest, çalıştırmak zorunlu. Takıldığın yerde önce kodun yorumlarını oku.

## 1 · PyTorch nedir, neden var?

Derin öğrenme modeli, arka planda **milyonlarca çarpma-toplama** yapan bir matematik makinesidir. Bu makineyi saf Python döngüleriyle yazmak hem yavaş hem hataya açıktır. PyTorch üç işi üstlenir:

| İş | Ne demek | Saf Python'da karşılığı |
|---|---|---|
| **Tensor** | Çok boyutlu sayı dizisi; GPU'da da çalışır | `list` / `numpy.ndarray` |
| **Otomatik türev (`autograd`)** | Her işlemin türevini kendisi hesaplar | Zincir kuralını elle yazmak |
| **NN katmanları (`torch.nn`)** | Hazır `Linear`, `Conv2d`, `LSTM` blokları | Her katmanı sıfırdan yazmak |

PyTorch, 2017'de Facebook AI (bugünkü Meta AI) tarafından açık kaynak yayınlandı; araştırma ve endüstride en yaygın çerçevelerden biri. TensorFlow aynı işi yapan alternatiftir — bu kursta **PyTorch** kullanacağız.

**Bu hazırlık notunun işi:** Ders 1'de "tensor" kavramı derinlemesine kurulur; burada hedef ortamın çalıştığını görmek ve tensor ile ilk el sıkışmayı yapmaktır.

### Arka plandaki matematik

Kurulum kontrolünün matematiği yok — ama iki kavram kritik: **sürüm** ve **aygıt (device)**. PyTorch sürümleri arasında fonksiyon imzaları değişebilir; GPU kodu `torch.cuda.is_available()` `False` dönerse çöker. Bu yüzden her ders notu aygıt kontrolüyle başlar: kod CPU'da çalışmak zorunda, GPU varsa hızlanır.

> Bu blok ortam sağlığı kontrolü için gerekli: kurulumun doğru olduğunu ve hangi aygıtta çalıştığımızı ilk hücrede görürüz — sorun varsa ders başlamadan çözeriz.

In [1]:
# torch'un kendisi: tensor kütüphanesi; her şey bu modülden başlar.
# Yarar: bu notun tüm işlemleri `torch.` ile başlar.
import torch

# Kurulu PyTorch sürümünü yazar; sürümü bilmek uyumluluk ve hata raporu
# için şart (fonksiyonlar sürümler arası değişebiliyor).
print("PyTorch sürümü :", torch.__version__)
# GPU (NVIDIA CUDA) var mı diye sorar; False dönerse her şey CPU'da
# koşar. Yarar: ders kodları iki durumda da doğru çalışır.
print("GPU (CUDA)     :", torch.cuda.is_available())
# CUDA sürümünü yalnızca GPU varken okur; yoksa çökmesin diye if'li.
if torch.cuda.is_available():
    print("CUDA sürümü    :", torch.version.cuda)
    print("GPU adı        :", torch.cuda.get_device_name(0))
# Tüm hesaplar için ortak aygıtı seçer: GPU varsa "cuda", yoksa "cpu".
device = "cuda" if torch.cuda.is_available() else "cpu"
# Seçilen aygıtı yazar; sonraki oturumlardaki kodlar hep bu kalıbı kullanır.
print("Aygıt          :", device)

PyTorch sürümü : 2.14.0+cu130
GPU (CUDA)     : True
CUDA sürümü    : 13.0
GPU adı        : NVIDIA GeForce RTX 4050 Laptop GPU
Aygıt          : cuda


## 2 · Tensor: sayıların çok boyutlu kutusu

**Tensor**, PyTorch'un temel veri taşıyıcısıdır; boyut sayısına göre isim alır:

| Boyut (ndim) | Ad | Örnek | `shape` |
|---|---|---|---|
| 0 | skaler | $3.14$ | `()` |
| 1 | vektör | $[1, 2, 3]$ | `(3,)` |
| 2 | matris | $2 \times 2$ tablo | `(2, 2)` |
| 3 | 3B tensor | RGB görüntü | `(8, 8, 3)` |

Görüntü, metin, ses — hepsi sonunda tensor olur. Derin öğrenme = tensorlerin toplama, çarpma ve türev alma ile işlenmesidir. Her tensor iki kimlik taşır: **shape** (boyutlar) ve **dtype** (sayı tipi).

### Arka plandaki matematik

Matris çarpımının kuralı: $(m \times n) \cdot (n \times p) = m \times p$ — iç boyutlar eşit olmalı.

Elle örnek: $\begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix} \cdot \begin{bmatrix} 5 \\ 6 \end{bmatrix} = \begin{bmatrix} 1\cdot5{+}2\cdot6 \\ 3\cdot5{+}4\cdot6 \end{bmatrix} = \begin{bmatrix} 17 \\ 39 \end{bmatrix}$

Yani $17 = 1\times5 + 2\times6$ ve $39 = 3\times5 + 4\times6$ — çıktının her hücresi bir satır·sütun iç çarpımıdır.

> Bu blok tensor oluşturma ve boyut kavramı için gerekli: her şey önce tensor'a dönüşür — shape ve dtype okuyabilmek ilk beceri; ilerideki tüm şekil hataları buradan çözülür.

In [2]:
# torch.tensor: bir Python listesinden tensor üretir. dtype varsayılanı
# float32'dir: sinir ağları 32-bit float ile çalışır, tam sayı da float'a
# çevrilir.
skaler = torch.tensor(3.14)
# Tek boyutlu tensor (vektör): shape (3,) — üç elemanlı.
vektor = torch.tensor([1.0, 2.0, 3.0])
# İki boyutlu tensor (matris): 2 satır, 2 sütun.
matris = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
# Üç boyutlu tensor: 2 "sayfa", her biri 2x2 matris.
tensor3 = torch.tensor([[[1.0, 2.0], [3.0, 4.0]],
                        [[5.0, 6.0], [7.0, 8.0]]])
# .shape boyutların listesini, .ndim kaç boyut olduğunu, .dtype sayı
# tipini verir. Yarar: "verim ne şekildedir?" sorusu her hata
# ayıklamanın ilk adımıdır — önce shape yazdır.
for ad, t in [("skaler", skaler), ("vektör", vektor),
              ("matris", matris), ("3B tensor", tensor3)]:
    print(f"{ad:>10} | shape={tuple(t.shape)} | ndim={t.ndim} | dtype={t.dtype}")

# Hazır üreticiler: elle yazmak yerine hızlı boş kap üretir.
sifir = torch.zeros(2, 3)          # 2x3 sıfır matrisi (ağırlık başlangıcı)
birler = torch.ones(4)             # dört elemanlı birler vektörü
aralik = torch.arange(0, 10, 2)    # 0,2,4,6,8 — eşit aralıklı tam sayılar
esit = torch.linspace(0.0, 1.0, 5) # 0 ile 1 arasında 5 eşit nokta
# Yarar: arange/linspace ders boyunca ızgara ve zaman adımı üretir.
print("zeros  :", sifir)
print("arange :", aralik, "| linspace:", esit)

    skaler | shape=() | ndim=0 | dtype=torch.float32
    vektör | shape=(3,) | ndim=1 | dtype=torch.float32
    matris | shape=(2, 2) | ndim=2 | dtype=torch.float32
 3B tensor | shape=(2, 2, 2) | ndim=3 | dtype=torch.float32
zeros  : tensor([[0., 0., 0.],
        [0., 0., 0.]])
arange : tensor([0, 2, 4, 6, 8]) | linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


## 3 · Tensor işlemleri ve broadcasting

Aynı shape'te iki tensorde toplama/çarpma **eleman eleman** yapılır. Shape'ler farklıysa **broadcasting** devreye girer: boyutlar sağdan sola hizalanır; her boyut ya **eşit** olmalı ya taraflardan biri **1** olmalı (o eksen sanal olarak kopyalanır).

Elle örnek — $[2, 3]$ matris + $[3]$ vektör:
$$\begin{bmatrix} 1 & 2 & 3 \\ 10 & 20 & 30 \end{bmatrix} + \begin{bmatrix} 1 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 2 & 3 & 4 \\ 11 & 21 & 31 \end{bmatrix}$$

Matris çarpımı (`@`) eleman eleman çarpma **değildir**: $c_{ij} = \sum_k a_{ik} b_{kj}$ — iç boyutlar eşit, sonuç dış boyutlarda.

### Arka plandaki matematik

Eleman eleman işlem: $c_{ij} = a_{ij} + b_{ij}$ — her konum bağımsız. Matris çarpımı: her çıktı hücresi bir satır·sütun iç çarpımıdır.

Elle örnek: $\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \cdot \begin{bmatrix} 5 \\ 7 \end{bmatrix} = \begin{bmatrix} 5 \\ 7 \end{bmatrix}$ — birim matris hiçbir şeyi değiştirmez; bu kontrol alttaki kodda `allclose` ile sayısal olarak yapılır.

Broadcasting kuralını sınamak: $(2, 3) + (3,) \to (2, 3)$ olur; $(2, 3) + (2,)$ **olmaz** — sağdan hizalamada 3 ≠ 2 ve ikisi de 1 olmadığı için reddedilir → kodda try/except ile görürsün.

> Bu blok işlem hücresi için gerekli: her model katmanı aslında matris çarpımı + eleman eleman işlemdir; broadcasting'in nerede patladığını bilmek ilerideki şekil hatalarını dakikalarda çözer.

In [3]:
# Aynı shape'te iki matris: eleman eleman işlem doğrudan çalışır.
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
# +, -, * : hepsi ELEMAN ELEMAN. a*b, matris çarpımı DEĞİLDİR!
print("a + b =", a + b)
print("a * b =", a * b, "  <- eleman eleman çarpım")
# @ işleci matris çarpımı: (2x2)·(2x2) → 2x2.
# Elle: c[0,0] = 1*10 + 2*30 = 70 ; c[0,1] = 1*20 + 2*40 = 100 ;
#       c[1,0] = 3*10 + 4*30 = 150 ; c[1,1] = 3*20 + 4*40 = 220.
matmul_sonuc = a @ b
print("a @ b =", matmul_sonuc)
# Elle hesapladığımız değerlerle karşılaştırma: allclose True dönerse
# kod ile elle hesap aynı demektir. Yarar: "matris çarpımı ne yapıyor?"
# sorusunun sayısal kanıtı.
elle = torch.tensor([[70.0, 100.0], [150.0, 220.0]])
print("Elle hesapla eşit mi :", torch.allclose(matmul_sonuc, elle))

# Broadcasting: (2x3) + (3,) → vektör satırlara sanal kopyalanır.
m = torch.tensor([[1.0, 2.0, 3.0], 
                  [10.0, 20.0, 30.0]])
v = torch.tensor([1.0, 1.0, 1.0])
print("broadcast toplam :", m + v)
# Karşı örnek: (2x3) + (2,) kurala AYKIRI (3 ≠ 2, ikisi de 1 değil) →
# hata üretir; try/except ile gösteriyoruz.
try:
    m + torch.tensor([10.0, 20.0])
    print("Bu satır hiç çalışmaz")
except RuntimeError as hata:
    print("Broadcast hatası (bekleniyor):", str(hata)[:60], "...")

a + b = tensor([[11., 22.],
        [33., 44.]])
a * b = tensor([[ 10.,  40.],
        [ 90., 160.]])   <- eleman eleman çarpım
a @ b = tensor([[ 70., 100.],
        [150., 220.]])
Elle hesapla eşit mi : True
broadcast toplam : tensor([[ 2.,  3.,  4.],
        [11., 21., 31.]])
Broadcast hatası (bekleniyor): The size of tensor a (3) must match the size of tensor b (2) ...


## 4 · Numpy ↔ Torch köprüsü

Kursta bazı veri üretimleri NumPy ile, eğitim torch ile olur; köprü üç işlevden ibaret:

| Yön | Fonksiyon | Not |
|---|---|---|
| numpy → torch | `torch.from_numpy(dizi)` | Belleği **paylaşır**: birini değiştirirsen diğeri de değişir! |
| torch → numpy | `t.numpy()` | Yalnız CPU tensorunda çalışır |
| skaler çıkarma | `t.item()` | Tek elemanlı tensoru Python sayısına çevirir |

### Arka plandaki matematik

Bu bölümde yeni matematik yok — bilgisayarda **bellek paylaşımı** var: `from_numpy` yeni bellek ayırmaz, numpy dizisinin belleğini sarmalar. Bu yüzden bağımsız kopya istenirse `torch.tensor(dizi)` veya `t.clone()` kullanılır.

> Bu blok köprü hücresi için gerekli: ders boyunca veri üretimi çoğunlukla numpy ile (OpenCV, sklearn, PIL) olur; modele vermeden önce torch'a çevrilir — köprünün maliyetini ve tuzağını bilmek şart.

In [4]:
# numpy'i getirir: klasik sayı dizisi kütüphanesi; veri üretiminde ve
# OpenCV/sklearn ile köprüde kullanılır.
import numpy as np

# numpy dizisi üretir: 0..5 arası 6 nokta.
dizi = np.arange(6, dtype=np.float32)
# from_numpy BELLEĞİ PAYLAŞIR: yeni bellek ayırmaz, dizi ile t aynı
# veriyi gösterir. Yarar: kopyasız, hızlı geçiş.
t = torch.from_numpy(dizi)
# Tensoru yazdırır: numpy ile torch aynı sayıları gösterir.
print("numpy:", dizi, "| torch:", t)
# Bir tarafı değiştirir; paylaşılan bellek yüzünden ÖBÜRÜ de değişir!
dizi[0] = 99.0
# Kanıt: t[0] artık 99.0 — paylaşımlı belleğin işareti.
print("Paylaşımlı mı? dizi[0]=99 iken t[0] =", t[0].item())
# item(): tek elemanlı tensoru düz Python sayısına çevirir; print ve
# koşullarda tensor yerine sayı kullanılır.
print("t[1] bir Python sayısı:", t[1].item(), type(t[1].item()).__name__)
# numpy(): CPU tensorunu numpy dizisine döndürür; bağımsız kopya için
# önce clone() alır (paylaşım tuzağına düşmeden).
yeni = t.clone().numpy()
yeni[1] = -1.0
# Kanıt: clone sonrası iki taraf birbirinden bağımsız; t değişmedi.
print("yeni[1] değişti ama t[1] =", t[1].item(), "(bağımsız)")

numpy: [0. 1. 2. 3. 4. 5.] | torch: tensor([0., 1., 2., 3., 4., 5.])
Paylaşımlı mı? dizi[0]=99 iken t[0] = 99.0
t[1] bir Python sayısı: 1.0 float
yeni[1] değişti ama t[1] = 1.0 (bağımsız)


## 5 · Otomatik türev: autograd'a ilk bakış

Ders 2'nin konusu gradyan inişi; orada türevlere **elle** bakacağız. PyTorch'un süper gücü: türevleri **kendisi** hesaplar. Kural: hesapladığın sayıya "izle" (`requires_grad=True`) der, işlemler bitince `backward()` çağırırsın — PyTorch türevleri `.grad` alanına yazar.

### Arka plandaki matematik

Türev = fonksiyonun o noktadaki eğimi. Tek örnek: $f(x) = x^2 \Rightarrow f'(x) = 2x$. $x = 3$'te eğim $6$ — $x$'i küçük miktarda artırırsan $f$ altı katı artar.

İç içe fonksiyonda **zincir kuralı** devreye girer: $f(x) = (3x + 2)^2$ ile $u = 3x + 2$ dersek $f'(x) = f'(u) \cdot u'(x) = 2u \cdot 3 = 6(3x+2)$. $x = 1$ için $f'(1) = 6 \cdot 5 = 30$. PyTorch bu zinciri otomatik kurar ve çarpar — bize sadece `backward()` yazmak kalır.

İkinci örnek: $L(w) = (w \cdot x - y)^2$ için $\frac{dL}{dw} = 2(wx - y) \cdot x$; $x=1, y=2, w=0$ için $2(0-2)\cdot1 = -4$.

> Bu blok autograd hücresi için gerekli: otomatik türev, Ders 2'de kurulacak gradyan inişinin motorudur — burada sadece "PyTorch türevleri kendisi hesaplıyor" kanıtını görüyoruz.

In [5]:
# requires_grad=True: PyTorch'a "bu değişkeni izle, türevini tut" der.
# Yarar: modelin AĞIRLIKLARI hep böyle işaretlenir; girdiler izlenmez.
x = torch.tensor(1.0, requires_grad=True)
# İç içe fonksiyon: f = (3x + 2)^2. backward() zincir kuralını otomatik
# uygular: f'(x) = 2(3x+2) * 3 = 6(3x+2); x=1 için 30.
f = (3 * x + 2) ** 2
# Türevleri hesaplar; sonuç x.grad alanına yazılır.
f.backward()
# x.grad'i okur: elle hesapla 30 eşleşmeli. Yarar: Ders 2'de elle
# gradyan ile autograd karşılaştırılacak — altyapı burada hazır.
print("f  =", f.item(), "| f'(x) =", x.grad.item(), " (elle: 30)")

# İkinci örnek: basit doğrusal model y = w*x üzerinde w'nin kayba
# etkisi — Ders 2'nin birebir ön haberi.
w = torch.tensor(0.0, requires_grad=True)
# Tek veri noktası: gerçek ilişki y = 2x, tahminimiz w*x.
x2 = torch.tensor(1.0)
hedef = torch.tensor(2.0)
# Kayıp = (tahmin - hedef)^2 : modelin ne kadar yanıldığı.
kayip = (w * x2 - hedef) ** 2
# Türev hesaplanır: dL/dw = 2(w*x - hedef)*x = 2(0-2)*1 = -4.
kayip.backward()
# Gradyanı yazar: -4.0 — yani w'yi ARTIRIRSAN kayıp düşer (Ders 2'de
# w <- w - lr*gradyan kuralının temeli).
print("dL/dw =", w.grad.item())

f  = 25.0 | f'(x) = 30.0  (elle: 30)
dL/dw = -4.0


## 6 · Aygıt yönetimi: GPU varsa hızlan

Tensorlar hangi aygıtta (CPU/GPU) ise işlem orada yapılır; iki tensor farklı aygıttaysa hata alırsın. Ders kodları hep şu kalıpla aygıt bağımsız yazılır: `X.to(device)`.

### Arka plandaki matematik

Bu bölümün matematiği yok; amaç **aygıt tutarlılığı** kuralı: işlemdeki her tensor aynı aygıtta olmalı. GPU matris çarpımını binlerce çekirdekte paralel yapar; küçük veride CPU da yeterlidir. Kursta GPU garantisi yoktur — kodlar CPU'da çalışmak zorundadır, GPU bonusdur.

> Bu blok aygıt hücresi için gerekli: `.to(device)` kalıbını görmek — ders boyunca her veri ve model bu tek satırla aygıta taşınır.

In [6]:
# Aygıtı seçer: GPU varsa "cuda", yoksa "cpu" — kod iki durumda da
# çalışır. Yarar: katılımcının makinesi ne olursa olsun hata yok.
device = "cuda" if torch.cuda.is_available() else "cpu"
# 2x2 matris üretir ve seçilen aygıta taşır; .to(device) tensorun
# aygıtını değiştirir (veri GPU'ya kopyalanır).
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]]).to(device)
# Tensorun şu an hangi aygıtta olduğunu yazar: "cpu" veya "cuda:0".
# Yarar: "iki tensor farklı aygıtta" hatasını önlemenin ilk kontrolü.
print("t aygıtı :", t.device)
# Aygıttan bağımsız toplama: iki taraf da aynı aygıtta, işlem sorunsuz.
print("t + 1    :", t + 1)
# Not: GPU yoksa bu hücre zaten CPU'da koştu — hiçbir şey bozulmaz.
print("Aygıt:", device, "| işlem tamam")

t aygıtı : cuda:0
t + 1    : tensor([[2., 3.],
        [4., 5.]], device='cuda:0')
Aygıt: cuda | işlem tamam


## Mini Quiz

1. `torch.tensor([[1, 2], [3, 4]])` tensorünün `shape`'i nedir? `ndim`'i kaçtır?
2. `a @ b` ile `a * b` arasındaki fark nedir? `(2x3) @ (3x4)` işleminin sonucunun `shape`'i ne olur?
3. Broadcasting kuralına göre `shape=(2, 3)` bir matrise `shape=(3,)` bir vektör eklenebilir mi? `(2,)` vektör eklenebilir mi?
4. `torch.from_numpy(dizi)` ile üretilen tensoru değiştirirsen numpy dizisi ne olur?
5. `x = torch.tensor(2.0, requires_grad=True)` ve `f = x**3` için `f.backward()` sonrası `x.grad` kaçtır? (Elle: $f'(x) = 3x^2$)

### Cevaplar

<details>
<summary>Cevapları göster</summary>

1. `shape = (2, 2)`; `ndim = 2` (matris).
2. `a @ b` matris çarpımı (satır·sütun iç çarpımı), `a * b` eleman eleman çarpım. `(2x3) @ (3x4) = (2, 4)` — iç boyutlar eşit, sonuç dış boyutlarda.
3. `(3,)` vektör: **evet**, sağdan hizalamada 3 = 3 → satır otomatik genişler. `(2,)`: **hayır**, sağdan hizalamada 3 ≠ 2 ve ikisi de 1 değil → RuntimeError.
4. Numpy dizisi **de değişir** — `from_numpy` belleği paylaşır; bağımsız kopya için `clone()` gerekir.
5. $f'(x) = 3x^2 = 12$; autograd da `x.grad = 12.` döndürür.
</details>

## Özet

- **PyTorch** = tensor + otomatik türev + hazır katmanlar; derin öğrenmenin makinesi.
- Her veri önce **tensor** olur; `shape` ve `dtype` ilk kontrol yeridir.
- `@` matris çarpımı, `*` eleman eleman; **broadcasting** sağdan hizalanır (eşit veya 1).
- `from_numpy` belleği **paylaşır** — bağımsız kopya için `clone()`.
- `requires_grad=True` + `backward()` türevleri `.grad`'e yazar — Ders 2'nin motoru.

**Sonraki oturum — Ders 1: DL vs ML ve Tensor Kavramı.** Tensor'ı bugün tanıdık; Ders 1'de makine öğrenmesi ile derin öğrenmenin farkını, veriden tensöre giden yolu ve kuvvet yasasını kuracağız.

## Kaynakça

- PyTorch resmî sitesi: <https://pytorch.org>
- PyTorch Tutorials — "PyTorch 60 Minute Blitz": <https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html>
- Tensor dokümantasyonu: <https://pytorch.org/docs/stable/tensors.html>
- PyTorch tutorials repo: <https://github.com/pytorch/tutorials>
- 3Blue1Brown — "Neural networks" serisi (görsel giriş): <https://www.3blue1brown.com/topics/neural-networks>

## Kodlama Soruları

Bu hazırlık notunun kodları üzerinden 5 kodlama görevi. Her sorunun beklenen çıktısı tanımlıdır; önce kendin yazmayı dene, sonra çözüm taslağını aç.

**Soru 1 — Boyut merdiveni.** `torch.arange(24)` ile 0..23 vektörünü üret; `.reshape()` ile sırayla `(2, 12)`, `(2, 3, 4)`, `(2, 2, 3, 2)` şekillerine getir ve her aşamada `shape` + `ndim` yazdır. Beklenen: her aşamada eleman sayısı 24 kalır (`numel()` ile kanıtla).

<details><summary>Çözüm fikri</summary>

```python
import torch
t = torch.arange(24)
for sekil in [(2, 12), (2, 3, 4), (2, 2, 3, 2)]:
    t = t.reshape(*sekil)
    print(sekil, "->", tuple(t.shape), "| ndim:", t.ndim, "| eleman:", t.numel())
```
`numel()` her aşamada 24 — reshape yalnızca yorumlamayı değiştirir, veriyi değil.
</details>

**Soru 2 — Matris çarpımını elle doğrulamak.** `A = torch.tensor([[2.0, 0.0], [1.0, 3.0]])` ve `B = torch.tensor([[1.0, 4.0], [2.0, 5.0]])` için `A @ B`'yi önce elle hesapla (her hücreyi yaz), sonra kodla `allclose` ile doğrula. Beklenen elle sonuç: `[[2, 8], [7, 19]]`.

<details><summary>Çözüm fikri</summary>

Elle: `c[0,0] = 2*1 + 0*2 = 2`; `c[0,1] = 2*4 + 0*5 = 8`; `c[1,0] = 1*1 + 3*2 = 7`; `c[1,1] = 1*4 + 3*5 = 19`.
```python
print(A @ B)
print(torch.allclose(A @ B, torch.tensor([[2.0, 8.0], [7.0, 19.0]])))  # True
```
</details>

**Soru 3 — Broadcasting avcısı.** Aşağıdaki çiftlerden hangisi çalışır, hangisi RuntimeError verir? Önce kâğıtta kural uygula, sonra kodla dene: (a) `(4, 1) + (1, 3)`, (b) `(4, 3) + (4,)`, (c) `(2, 3, 4) + (4,)`.

<details><summary>Çözüm fikri</summary>

```python
a = torch.ones(4, 1); b = torch.ones(1, 3); c = torch.ones(4, 3)
d = torch.ones(2, 3, 4)
print((a + b).shape)   # (4, 3): boyutlar 1<->4 ve 1<->3 genişler → çalışır
print((c + torch.ones(4)).shape)  # hata: sağdan hizalamada 3 ≠ 4
print((d + torch.ones(4)).shape)  # (2, 3, 4): son boyut 4 = 4 → çalışır
```
(a) çalışır → `(4, 3)`; (b) çalışmaz (RuntimeError); (c) çalışır → `(2, 3, 4)`.
</details>

**Soru 4 — Paylaşım tuzağını yakalamak.** `d = np.array([1.0, 2.0])`; `t = torch.from_numpy(d)`; `t[0] = 100` sonrası `d[0]` kaçtır? Aynı deneyi `t = torch.tensor(d)` için tekrarla. Beklenen: ilkinde `d[0] = 100`, ikincisinde `d[0] = 1.0`.

<details><summary>Çözüm fikri</summary>

```python
import numpy as np
d = np.array([1.0, 2.0]); t = torch.from_numpy(d)
t[0] = 100
print(d)  # [100. 2.] — bellek paylaşılır, d de değişti!
d2 = np.array([1.0, 2.0]); t2 = torch.tensor(d2)
t2[0] = 100
print(d2)  # [1. 2.] — torch.tensor yeni bellek ayırır, bağımsız.
```
Kural: hız istiyorsan `from_numpy` (paylaşımlı), güvenlik istiyorsan `torch.tensor` veya `clone()`.
</details>

**Soru 5 — Autograd ile doğrulama.** `x = torch.tensor(2.0, requires_grad=True)` ile $f(x) = x^3 + 2x$ fonksiyonunun türevini autograd ile hesapla; elle $f'(x) = 3x^2 + 2 \Rightarrow f'(2) = 14$ ile karşılaştır. Beklenen: `x.grad == 14.0`.

<details><summary>Çözüm fikri</summary>

```python
x = torch.tensor(2.0, requires_grad=True)
f = x**3 + 2 * x
f.backward()
print(x.grad.item())  # 14.0
```
Toplam kuralı: $d(x^3)/dx = 3x^2$, $d(2x)/dx = 2$ → $6x + 2$; $x = 2$ → 14. autograd aynı sonucu verir.
</details>